In [1]:
import sys
import os

project_root = os.path.abspath("../../")
if project_root not in sys.path:
    sys.path.append(project_root)


In [2]:
import pandas as pd
from data_mining import Dataset
from sklearn.model_selection import train_test_split
from damavand.damavand.utils import z_score_scaler
from experiments.supervised_anomaly_detection.classifier import VibrationFeatureExtractor, ContrastiveModel
from experiments.supervised_anomaly_detection.utils import ContrastiveDataset, VibrationClassificationDataset, create_balanced_contrastive_pairs, contrastive_loss, evaluate_model
import torch
from torch import nn
from torch.utils.data import DataLoader

In [3]:
config = {
    "do_contrastive_pretraining": True,  # Set to False to skip contrastive representation learning
    "downsample_majority_class": True,  # Set to False to use the full, imbalanced training set
    "contrastive_epochs": 10,
    "classification_epochs": 15,
    "contrastive_lr": 0.001,
    "classification_lr": 0.0005,
    "batch_size_contrastive": 128,
    "batch_size_classify": 32,
}

In [4]:
dataset = Dataset(operations=["00", "01", "03", "04", "05", "06", "07", "08", "10", "11", "12", "14"])
data = dataset.mine(win_len=1000, hop_len=1000)

/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_000.h5  ---  (268288, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_001.h5  ---  (268288, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_003.h5  ---  (268288, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_004.h5  ---  (264192, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_005.h5  ---  (268288, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_006.h5  ---  (268288, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_007.h5  ---  (269312, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_008.h5  ---  (268288, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_M

In [5]:
df = pd.concat(data["OP05"]["0"])
df

,0,1,2,3,4,5,6,7,8,9,...,994,995,996,997,998,999,machine,operation,state,file
0,-9.0,-50.0,-48.0,11.0,17.0,-11.0,-1.0,9.0,19.0,-29.0,...,33.0,37.0,40.0,33.0,19.0,21.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
1,-3.0,27.0,-27.0,-39.0,7.0,9.0,54.0,33.0,23.0,-35.0,...,1188.0,1073.0,1138.0,897.0,1128.0,839.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
2,1007.0,630.0,942.0,595.0,897.0,616.0,650.0,528.0,638.0,452.0,...,439.0,610.0,616.0,757.0,550.0,718.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
3,519.0,573.0,368.0,435.0,203.0,199.0,187.0,54.0,11.0,-97.0,...,95.0,48.0,-39.0,-91.0,-27.0,-58.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
4,35.0,99.0,-17.0,33.0,37.0,3.0,-52.0,-48.0,-33.0,-5.0,...,11.0,1.0,-23.0,15.0,25.0,29.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,-7.0,-25.0,13.0,-37.0,5.0,-11.0,-5.0,-21.0,0.0,-15.0,...,-15.0,-17.0,-13.0,-15.0,-9.0,-11.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5
43,-15.0,-17.0,-7.0,-13.0,-13.0,-21.0,-11.0,-3.0,-5.0,-15.0,...,-11.0,-5.0,-37.0,-7.0,-27.0,-7.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5
44,-7.0,-31.0,-13.0,-33.0,-17.0,-31.0,-3.0,-44.0,-7.0,-21.0,...,-21.0,-13.0,-11.0,-21.0,-17.0,-15.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5
45,-17.0,-7.0,-17.0,-5.0,-15.0,-5.0,-15.0,-15.0,-19.0,-15.0,...,-13.0,-5.0,-21.0,-19.0,0.0,-29.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5


In [6]:
df_train, df_test = train_test_split(df, test_size=0.4, random_state=42)
df_train["state_encoded"] = df_train["state"].map({"good": 0, "bad": 1})
df_test["state_encoded"] = df_test["state"].map({"good": 0, "bad": 1})

In [7]:
if config["downsample_majority_class"]:
    print("Applying majority class downsampling to the training set...")
    # Separate classes based on encoded state
    df_good = df_train[df_train["state_encoded"] == 0]
    df_bad = df_train[df_train["state_encoded"] == 1]

    # Identify minority and majority sizes
    minority_size = min(len(df_good), len(df_bad))

    # Downsample both to match the minority size
    df_good_downsampled = df_good.sample(n=minority_size, random_state=42)
    df_bad_downsampled = df_bad.sample(n=minority_size, random_state=42)

    # Recombine
    df_train = pd.concat([df_good_downsampled, df_bad_downsampled]).sample(
        frac=1, random_state=42
    )
    print(f"New training set size: {len(df_train)} (Balanced)")
else:
    print("Using full, imbalanced training set.")


Applying majority class downsampling to the training set...
New training set size: 152 (Balanced)


In [8]:
x_train, y_train = df_train.iloc[:, :-5], df_train["state_encoded"]
x_test, y_test = df_test.iloc[:, :-5], df_test["state_encoded"]

x_train_scaled, x_test_scaled = z_score_scaler(x_train), z_score_scaler(x_test)
print(f"Train shape: {x_train_scaled.shape} | Test shape: {x_test_scaled.shape}")

# Verify final distribution
print("\nTrain class distribution:\n", y_train.value_counts())


Train shape: (152, 1000) | Test shape: (622, 1000)

Train class distribution:
 state_encoded
0    76
1    76
Name: count, dtype: int64


In [9]:
encoder = VibrationFeatureExtractor()
model = ContrastiveModel(encoder)

In [10]:
if config["do_contrastive_pretraining"]:
    print("\n>>> Starting Contrastive Representation Learning Loop...")
    X1, X2, Y_pairs = create_balanced_contrastive_pairs(x_train_scaled, y_train)
    contrastive_dataset = ContrastiveDataset(X1, X2, Y_pairs)
    contrastive_dataloader = DataLoader(
        contrastive_dataset, batch_size=config["batch_size_contrastive"], shuffle=True
    )

    model.set_mode("contrastive")
    optimizer_con = torch.optim.Adam(model.parameters(), lr=config["contrastive_lr"])

    for epoch in range(config["contrastive_epochs"]):
        epoch_loss = 0.0
        for x1_batch, x2_batch, y_batch in contrastive_dataloader:
            optimizer_con.zero_grad()
            z1 = model(x1_batch)
            z2 = model(x2_batch)
            loss = contrastive_loss(z1, z2, y_batch)
            loss.backward()
            optimizer_con.step()
            epoch_loss += loss.item()

        print(
            f"Contrastive Epoch [{epoch + 1}/{config['contrastive_epochs']}] - Loss: {epoch_loss / len(contrastive_dataloader):.4f}"
        )
else:
    print("\n>>> Skipping Contrastive Pretraining (disabled in config).")



>>> Starting Contrastive Representation Learning Loop...
Contrastive Epoch [1/10] - Loss: 0.2180
Contrastive Epoch [2/10] - Loss: 0.0397
Contrastive Epoch [3/10] - Loss: 0.0083
Contrastive Epoch [4/10] - Loss: 0.0060
Contrastive Epoch [5/10] - Loss: 0.0033
Contrastive Epoch [6/10] - Loss: 0.0032
Contrastive Epoch [7/10] - Loss: 0.0023
Contrastive Epoch [8/10] - Loss: 0.0024
Contrastive Epoch [9/10] - Loss: 0.0018
Contrastive Epoch [10/10] - Loss: 0.0018


In [11]:
train_classify_dataset = VibrationClassificationDataset(x_train_scaled, y_train)
train_loader = DataLoader(
    train_classify_dataset, batch_size=config["batch_size_classify"], shuffle=True
)

model.set_mode("classify")
criterion = nn.CrossEntropyLoss()
optimizer_cls = torch.optim.Adam(model.parameters(), lr=config["classification_lr"])

model.train()
print("\n>>> Starting Downstream Classification Training...")
for epoch in range(config["classification_epochs"]):
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer_cls.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_cls.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(
        f"Classification Epoch [{epoch + 1}/{config['classification_epochs']}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%"
    )



>>> Starting Downstream Classification Training...
Classification Epoch [1/15] - Loss: 0.7005 - Accuracy: 52.63%
Classification Epoch [2/15] - Loss: 0.6477 - Accuracy: 63.82%
Classification Epoch [3/15] - Loss: 0.6024 - Accuracy: 75.66%
Classification Epoch [4/15] - Loss: 0.5610 - Accuracy: 82.89%
Classification Epoch [5/15] - Loss: 0.5250 - Accuracy: 88.16%
Classification Epoch [6/15] - Loss: 0.4879 - Accuracy: 86.18%
Classification Epoch [7/15] - Loss: 0.4426 - Accuracy: 89.47%
Classification Epoch [8/15] - Loss: 0.4079 - Accuracy: 91.45%
Classification Epoch [9/15] - Loss: 0.3705 - Accuracy: 92.76%
Classification Epoch [10/15] - Loss: 0.3400 - Accuracy: 95.39%
Classification Epoch [11/15] - Loss: 0.3034 - Accuracy: 93.42%
Classification Epoch [12/15] - Loss: 0.2907 - Accuracy: 93.42%
Classification Epoch [13/15] - Loss: 0.2502 - Accuracy: 95.39%
Classification Epoch [14/15] - Loss: 0.2409 - Accuracy: 94.08%
Classification Epoch [15/15] - Loss: 0.2153 - Accuracy: 95.39%


In [12]:
metrics = evaluate_model(model, x_test_scaled, y_test, target_names=["good", "bad"], config=config, export_path="results.txt")


============= Experiment configuration =============
config:
  do_contrastive_pretraining: True
  downsample_majority_class: True
  contrastive_epochs: 10
  classification_epochs: 15
  contrastive_lr: 0.001
  classification_lr: 0.0005
  batch_size_contrastive: 128
  batch_size_classify: 32

================ Evaluation Metrics ================
              precision    recall  f1-score   support

        good       0.99      0.96      0.98       584
         bad       0.61      0.89      0.72        38

    accuracy                           0.96       622
   macro avg       0.80      0.93      0.85       622
weighted avg       0.97      0.96      0.96       622

Confusion Matrix:
True Good: 562 | False Bad: 22
False Good: 4 | True Bad: 34

--> Performance report successfully saved to: results.txt
